# GLiNER PII Detection Example

This notebook shows how to use GLiNER for PII detection and PII masking in NeMo Guardrails.

GLiNER is a Generalist and Lightweight Model for Named Entity Recognition that can detect a wide range of entity types, including comprehensive PII categories.


## Local Deployment

Pull and run both NIM containers. You need an **NGC Personal API key** to pull these images and let each NIM download its model artifacts — generate one at [org.ngc.nvidia.com/setup/api-keys](https://org.ngc.nvidia.com/setup/api-keys) with at least the **NGC Catalog** service selected. Export it as `NGC_API_KEY`:

```bash
export NGC_API_KEY="<your-ngc-key>"
```

This is a different key from the `NVIDIA_API_KEY` you set earlier — they authenticate against different services (NGC for image pulls and model downloads vs. `integrate.api.nvidia.com` for hosted inference) even though both are issued by NVIDIA.

On a multi-GPU host, pin each container to a distinct GPU with `--gpus '"device=N"'` instead of `--gpus all`. Without an explicit device, both NIMs default to GPU 0 and compete for memory. The examples below assign GLiNER to GPU 0 and Llama to GPU 1; adjust the indices to match your host.

**GLiNER-PII NIM** (port 8000, GPU 0):

```bash
# Authenticate with NGC (username: $oauthtoken, password: your NGC API key)
echo $NGC_API_KEY | docker login -u '$oauthtoken' --password-stdin nvcr.io

docker run --rm -it --gpus '"device=0"' \
  -e NGC_API_KEY \
  -p 8000:8000 \
  nvcr.io/nim/nvidia/gliner-pii:1.0.0-rc1
```

**Llama 3.1 8B Instruct NIM** (port 8001, GPU 1):

```bash
docker run --rm -it --gpus '"device=1"' \
  -e NGC_API_KEY \
  -p 8001:8000 \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

Wait until both containers log `Application startup complete`, then set `DEPLOYMENT = 'local'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Remote Deployment

Set your NVIDIA API key before running the config cells:

```bash
export NVIDIA_API_KEY="nvapi-..."
```

You can obtain an API key at [build.nvidia.com](https://build.nvidia.com).

`nvidia/gliner-pii` does not appear in the configs below because it is the default value of `rails.config.gliner.model`. You only need to set that field explicitly if you want to use a different model:

```yaml
rails:
  config:
    gliner:
      model: nvidia/gliner-pii  # default — omit or change as needed
```

Alternatively, you can add  

```python
    config.rails.config.gliner.model = 'nvidia/gliner-pii' # default — omit or change as needed
```

under the `elif DEPLOYMENT == 'remote'` statements in the config cells. 

Set `DEPLOYMENT = 'remote'` in the **Choose Deployment Type** cell below and run the remaining cells.

## Choose Deployment Type

Set `DEPLOYMENT` to `'local'` if you completed the **Local Deployment** setup above, or `'remote'` if you are using the NVIDIA-hosted endpoint.

In [1]:
# Choose local or remote deployment
DEPLOYMENT = "remote"
assert DEPLOYMENT == "local" or DEPLOYMENT == "remote", "DEPLOYMENT parameter must be set to 'local' or 'remote'"

## Import the Necessary Modules

In [2]:
import nest_asyncio

from nemoguardrails import LLMRails, RailsConfig

nest_asyncio.apply()

## PII Detection


### Input rails

When PII is detected in user input, the request is blocked.


In [3]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

rails:
  config:
    gliner:
      threshold: 0.5
      input:
        entities:
          - first_name
          - last_name
          - street_address
          - email
      output:
        entities:
          - first_name
          - last_name
          - street_address
          - email
  input:
    flows:
      - gliner detect pii on input

  output:
    flows:
      - gliner detect pii on output
"""


config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

# Flesh out the config with deployment-dependent values
# Note that the api_key_env_var parameters are pointers to an environmental variable name,
# not the key itself
if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.rails.config.gliner.server_endpoint = "http://localhost:8000/v1/chat/completions"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.rails.config.gliner.server_endpoint = "https://integrate.api.nvidia.com/v1/chat/completions"
    config.rails.config.gliner.api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

In [4]:
response = rails.generate(
    messages=[{"role": "user", "content": "Hello! I'm John. My email id is text@gmail.com. I live in California, USA."}]
)

info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])


print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
I'm sorry, I can't respond to that.


Colang history
----------------------------------------
execute gliner_detect_pii
# The result was True
bot refuse to respond
  "I'm sorry, I can't respond to that."
bot stop



LLM calls summary
----------------------------------------
No LLM calls were made.


### Output rails

When PII is detected in the LLM output, the response is blocked.


In [5]:
response = rails.generate(messages=[{"role": "user", "content": "give me a sample email id"}])

info = rails.explain()

print("Response")
print("----------------------------------------\n\n")
print(response["content"])


print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()


print("\n\nCompletions where PII was detected!")
print("----------------------------------------")
print(info.llm_calls[0].completion)

Response
----------------------------------------


I'm sorry, I can't respond to that.


Colang history
----------------------------------------
execute gliner_detect_pii
# The result was False
user "give me a sample email id"
execute gliner_detect_pii
# The result was True
bot refuse to respond
  "I'm sorry, I can't respond to that."
bot stop



LLM calls summary
----------------------------------------
Summary: 1 LLM call(s) took 1.79 seconds and used 315 tokens.

1. Task `general` took 1.79 seconds and used 315 tokens.



Completions where PII was detected!
----------------------------------------
I can certainly generate a sample email ID for you. A typical email ID is a combination of a username and a domain name, separated by the '@' symbol. 

For example, let's say we want to create an email ID for a fictional character named "JohnDoe." We could choose a domain name that is relevant to John's profession or interests. 

Here's a sample email ID: 

john.doe@techsupport.com

This 

## PII Masking

Instead of blocking requests that contain PII, the masking flow replaces detected entities with label placeholders before the text reaches the LLM. For example, `Hi, I'm John — john@example.com` becomes `Hi, I'm [FIRST_NAME] — [EMAIL]`.

### Input rails

Detected PII in the user message is masked before the LLM sees it.

In [6]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

rails:
  config:
    gliner:
      threshold: 0.5
      input:
        entities:
          - first_name
          - city
          - country
          - email
  input:
    flows:
      - gliner mask pii on input
"""


config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

# Flesh out the config with deployment-dependent values
# Note that the api_key_env_var parameters are pointers to an environmental variable name,
# not the key itself
if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.rails.config.gliner.server_endpoint = "http://localhost:8000/v1/chat/completions"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.rails.config.gliner.server_endpoint = "https://integrate.api.nvidia.com/v1/chat/completions"
    config.rails.config.gliner.api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

In [7]:
response = rails.generate(
    messages=[{"role": "user", "content": "Hello! I'm John. My email id is text@gmail.com. I live in California, USA."}]
)

info = rails.explain()

print("Response")
print("----------------------------------------")
print(response["content"])


print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()

Response
----------------------------------------
Nice to meet you, [FIRST_NAME]! It's great that you're taking the time to introduce yourself. I'll make sure to remember your email id, [EMAIL], for any future conversations we may have. 

As for your location, I see that you're from [CITY], [COUNTRY]. That's fascinating! [COUNTRY] is a beautiful country with a rich history and culture. I've been trained on a vast amount of data, and I must say, I'm intrigued by the unique blend of traditions and customs that exist in [COUNTRY].

If you're willing to share, what brings you here today? Are you looking for information on a specific topic or perhaps seeking some general advice? I'm all ears (or rather, all text) and ready to help in any way I can!


Colang history
----------------------------------------
execute gliner_mask_pii
# The result was Hello! I'm [FIRST_NAME]. My email id is [EMAIL]. I live in [CITY], [COUNTRY].
user "Hello! I'm [FIRST_NAME]. My email id is [EMAIL]. I live in [CIT

### Output rails

Detected PII in the LLM response is masked before it is returned to the user.

In [8]:
# For remote deployment:
# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

YAML_CONFIG = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.1-8b-instruct

rails:
  config:
    gliner:
      threshold: 0.5
      output:
        entities:
          - first_name
          - city
          - country
          - email
  output:
    flows:
      - gliner mask pii on output
"""


config = RailsConfig.from_content(yaml_content=YAML_CONFIG)

# Flesh out the config with deployment-dependent values
# Note that the api_key_env_var parameters are pointers to an environmental variable name,
# not the key itself
if DEPLOYMENT == "local":
    config.models[0].parameters["base_url"] = "http://localhost:8001/v1"
    config.rails.config.gliner.server_endpoint = "http://localhost:8000/v1/chat/completions"
elif DEPLOYMENT == "remote":
    config.models[0].api_key_env_var = "NVIDIA_API_KEY"
    config.rails.config.gliner.server_endpoint = "https://integrate.api.nvidia.com/v1/chat/completions"
    config.rails.config.gliner.api_key_env_var = "NVIDIA_API_KEY"

rails = LLMRails(config)

In [9]:
response = rails.generate(messages=[{"role": "user", "content": "give me a sample email id"}])

info = rails.explain()

print("Response")
print("----------------------------------------\n\n")
print(response["content"])


print("\n\nColang history")
print("----------------------------------------")
print(info.colang_history)

print("\n\nLLM calls summary")
print("----------------------------------------")
info.print_llm_calls_summary()


print("\n\nOriginal completion (before masking)")
print("----------------------------------------")
print(info.llm_calls[0].completion)

Response
----------------------------------------


I'd be happy to provide you with a sample email ID.

A sample email ID could be something like this: `[EMAIL]`. 

In this example:

- `john.doe` is the username or the name of the person who owns the email account.
- `@` is the symbol used to separate the username from the domain name.
- `example.com` is the domain name, which is the name of the email service provider or the organization that hosts the email account.

You can replace `[EMAIL]` with your own name or a combination of your name and initials, and `example.com` with a valid domain name such as `gmail.com`, `yahoo.com`, or `hotmail.com`.


Colang history
----------------------------------------
user "give me a sample email id"
execute gliner_mask_pii
# The result was I'd be happy to provide you with a sample email ID.  A sample email ID could be something like this: `[EMAIL]`.   In this example:  - `john.doe` is the username or the name of the person who owns the email acco

## Supported Entity Types

The NVIDIA GLiNER-PII NIM supports these PII categories:

| Category | Entity Types |
|----------|-------------|
| Personal Identifiers | `first_name`, `last_name`, `ssn`, `date_of_birth`, `age`, `gender` |
| Contact Information | `email`, `phone_number`, `fax_number`, `street_address`, `city`, `state`, `postcode`, `country`, `county` |
| Financial | `credit_debit_card`, `cvv`, `bank_routing_number`, `account_number`, `swift_bic`, `tax_id` |
| Technical | `ipv4`, `ipv6`, `mac_address`, `url`, `api_key`, `password`, `pin`, `http_cookie` |
| Identification | `national_id`, `license_plate`, `vehicle_identifier`, `employee_id`, `customer_id`, `unique_id` |
| Sensitive Attributes | `sexuality`, `political_view`, `race_ethnicity`, `religious_belief`, `blood_type` |
